# 03 - Scaling & Normalization
Transform numeric features into comparable scales for clustering, classification, and forecasting models.
Strategies: StandardScaler (z-score), MinMaxScaler ([0,1]), RobustScaler (median & IQR).
Input: `data/processed/02_*.csv` | Output: `data/processed/03_scaled_*.csv` with original + scaled columns.

In [1]:
import numpy as np, pandas as pd, warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT = Path.cwd().parent.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
datasets = {}
for fname in ['02_worldbank_features.csv', '02_wto_demand_features.csv', '02_diversification_metrics.csv']:
    path = PROCESSED_DIR / fname
    if path.exists():
        datasets[fname.replace('.csv', '')] = pd.read_csv(path)
        print(f'Loaded {fname}: {datasets[fname.replace(".csv", "")].shape}')
    else:
        print(f'Skipped {fname} (not found)')

Loaded 02_worldbank_features.csv: (5375, 13)
Loaded 02_wto_demand_features.csv: (181, 6)
Loaded 02_diversification_metrics.csv: (11, 4)


In [2]:
def get_numeric_cols(df, exclude=None):
    if exclude is None:
        exclude = ['year', 'refyear', 'country_code', 'reporteriso', 'partneriso', 'cmdcode', 'product_sector_code', 'indicator', 'source', 'flow_type']
    nums = df.select_dtypes(include=[np.number]).columns.tolist()
    return [c for c in nums if c not in exclude and not c.endswith(('_std', '_minmax', '_robust'))]
def scale_dataframe(df, numeric_cols, scaler, suffix):
    df_out = df.copy()
    available = [c for c in numeric_cols if c in df_out.columns and df_out[c].notna().any()]
    if not available:
        return df_out
    fill_vals = df_out[available].median()
    scaled_matrix = scaler.fit_transform(df_out[available].fillna(fill_vals))
    for i, col in enumerate(available):
        df_out[f'{col}_{suffix}'] = scaled_matrix[:, i]
    return df_out
scalers = {'std': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}
print('Scalers ready:', list(scalers.keys()))

Scalers ready: ['std', 'minmax', 'robust']


## 3.1 - Scale World Bank Macro Features

In [3]:
key = '02_worldbank_features'
if key in datasets:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print('World Bank numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled World Bank shape:', df.shape)
    scaled_cols = [c for c in df.columns if c.endswith(('_std', '_minmax', '_robust'))]
    display(df[['country', 'year'] + scaled_cols[:6]].head())
else:
    print('World Bank dataset not available.')

World Bank numeric columns: ['NE.IMP.GNFS.ZS', 'NY.GDP.MKTP.CD', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD', 'SP.POP.TOTL', 'TG.VAL.TOTL.GD.ZS', 'gdp_growth_lag1', 'trade_volume_proxy', 'import_volume_proxy', 'gdp_per_capita_computed']
Scaled World Bank shape: (5375, 43)


,country,year,NE.IMP.GNFS.ZS_std,NY.GDP.MKTP.CD_std,NY.GDP.MKTP.KD.ZG_std,NY.GDP.PCAP.CD_std,SP.POP.TOTL_std,TG.VAL.TOTL.GD.ZS_std
0,Aruba,2000.0,0.919685,-0.189228,0.729219,0.202958,-0.171943,5.205478
1,Aruba,2001.0,0.870979,-0.189218,0.134058,0.205330,-0.171941,4.714512
2,Aruba,2002.0,0.843551,-0.189190,-0.752728,0.228092,-0.171939,2.866225
3,Aruba,2003.0,0.896179,-0.189155,-0.397205,0.253868,-0.171937,3.837717
4,Aruba,2004.0,0.809596,-0.189066,0.672280,0.324152,-0.171933,6.176296


## 3.2 - Scale WTO Demand Features

In [4]:
key = '02_wto_demand_features'
if key in datasets and not datasets[key].empty:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print('WTO numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled WTO demand shape:', df.shape)
    scaled_cols = [c for c in df.columns if c.endswith(('_std', '_minmax', '_robust'))]
    display(df[['year', 'product_sector'] + scaled_cols[:6]].head())
else:
    print('WTO demand dataset not available or empty.')

WTO numeric columns: ['value', 'global_demand_index', 'demand_growth_rate_pct', 'demand_3yr_ma']
Scaled WTO demand shape: (181, 18)


,year,product_sector,value_std,global_demand_index_std,demand_growth_rate_pct_std,demand_3yr_ma_std,value_minmax,global_demand_index_minmax
0,2015,Agricultural Products,-0.214487,-0.223341,-0.044179,-0.213925,0.052525,0.063618
1,2016,Agricultural Products,-0.239687,-0.219434,-0.453352,-0.226791,0.047569,0.064546
2,2017,Agricultural Products,-0.234694,-0.227972,-0.023260,-0.229380,0.048551,0.062519
3,2018,Agricultural Products,-0.271225,-0.284482,-0.653807,-0.248692,0.041367,0.049105
4,2019,Agricultural Products,-0.259775,-0.250865,0.103021,-0.255529,0.043618,0.057085


## 3.3 - Scale Diversification Metrics

In [5]:
key = '02_diversification_metrics'
if key in datasets and not datasets[key].empty:
    df = datasets[key].copy()
    num_cols = get_numeric_cols(df)
    print(f'{key} numeric columns:', num_cols)
    for suffix, scaler in scalers.items():
        df = scale_dataframe(df, num_cols, scaler, suffix)
    datasets[key] = df
    print('Scaled shape:', df.shape)
else:
    print(f'{key} not available or empty.')

02_diversification_metrics numeric columns: ['hhi', 'shannon_entropy', 'shannon_entropy_norm']
Scaled shape: (11, 13)


## 3.4 - Save All Scaled Datasets

In [6]:
output_mapping = {
    '02_worldbank_features': '03_worldbank_scaled.csv',
    '02_wto_demand_features': '03_wto_demand_scaled.csv',
    '02_diversification_metrics': '03_diversification_scaled.csv'
}
for key, out_name in output_mapping.items():
    if key in datasets and not datasets[key].empty:
        out_path = PROCESSED_DIR / out_name
        datasets[key].to_csv(out_path, index=False)
        print(f'Saved {out_name} -> shape {datasets[key].shape}')
    else:
        print(f'Skipped {out_name} (source missing)')
print('\nScaling complete. Proceed to 04_master_integration_and_eda.ipynb.')

Saved 03_worldbank_scaled.csv -> shape (5375, 43)
Saved 03_wto_demand_scaled.csv -> shape (181, 18)
Saved 03_diversification_scaled.csv -> shape (11, 13)

Scaling complete. Proceed to 04_master_integration_and_eda.ipynb.
